<a href="https://colab.research.google.com/github/Mhernandezg/AlgoritmosOptimizacionVIU/blob/main/TRABAJO_PRACTICO/Trabajo_Pr%C3%A1ctico_Tomas_de_Doblaje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Maycol Hern+andez García  <br>
Url: https://github.com/Mhernandezg/AlgoritmosOptimizacionVIU/blob/main/TRABAJO_PRACTICO/Trabajo_Pr%C3%A1ctico_Tomas_de_Doblaje.ipynb<br>
Google Colab: https://colab.research.google.com/drive/1Xw1BYb1h3NExfwmv9IQaHbJEgPSRXq6F?usp=sharing <br>
Problema:
1. Sesiones de doblaje <br>

    Descripción del problema:(copiar enunciado)

    Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible.

    Número de actores: 10
    Número de tomas : 30
    Actores/Tomas : https://bit.ly/36D8IuK
    - 1 indica que el actor participa en la toma
    - 0 en caso contrario







                                        

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?



1.   El espacio de soluciones representa una asignación de cada toma de doblaje a un día de grabación. Por lo que se agruparán las tomas en conjuntos que equivalen a un día de trabajo. Esto acorde a la matriz brindada con los datos del problema. Donde se tienen 30 tomas a distintas sesiones de grabación, donde cada sesión equivale a un día de trabajo. En la implementación, una solución se modela como una lista de sesiones, y cada sesión contiene una lista de tomas. De esta forma, se puede saber con claridad qué tomas se realizan en cada día y evaluar posteriormente qué actores deben asistir a cada sesión.

2.   El problema al consistir en minimizar el costo total del doblaje de una película. Por lo que al considerar los costos diarios hay que validar la cantidad de actores que asistieron durante el día.
El coste diario se calcula como el número de actores distintos que participan en al menos una de las tomas asignadas a la sesión

3.  Cada toma es asignable a un día. Todas las tomas deben ser asignadas. Y por último, cada día debe tener un máximo de 6 tomas.



#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

El problema establece una optimizacion fundamentada en combinaciones. Donde al tener 30 tomas distribuidas en diferentes sesiones de grabación acorde a la disponibilidad de los actores de voz. Se espera obtener la combinación de tomas segmentadas en diversos dias segun la restricción de maximo 6 tomas por dia.

En el codigo implementado para la solucion hablamos de escenarios donde se calcula una posible solucion local y se compara con todas las posibles soluciones aledañas. Lo que implica una validacion casi exponencial a medida que se realizan las iteraciones dentro de la busqueda local respecto a sus vecinos.

Y acorde a lo visto durante el curso se tiene presente que cuando no es posible encontrar un algoritmo realmente eficiente o práctico para alcanzar el óptimo, deben emplearse métodos heurísticos que busquen buenas soluciones mediante iteraciones y mejoras sucesivas.

Por lo que este escenario cumple dicha premisa.



#Diseño
- ¿Que técnica utilizo? ¿Por qué?

Se planteo una tecnica fundamentada en la heuristica debido a la complejidad del problema y el tamaño posible de soluciones que pueden existir.

Por lo que a manera de reto se opto por implementar la tecnica GRASP, es decir, un procedimiento de búsqueda voraz aleatorio y adaptativo. Mas que todo para establecer un precedente de implementacion de costos y combinaciones posibles acorde a escenarios puntuales. Algo que en un futuro TFM puede llegar a tener gran utilidad.

GRASP permite construir soluciones de forma iterativa y mejorar constantemente cada iteracion por medio de la busqueda local. El manual de la asignatura describe esta metaheurística como un proceso multiarranque compuesto por una fase de construcción y una fase de mejora local, exactamente el esquema que se ha seguido en la implementación.

En la fase constructiva, el algoritmo genera una solución sesión por sesión. toma como punto inicial una toma aleatoria desde la cual se van incorporando nuevas tomas a la sesión que agrupa una cantidad maxima de tomas. Para elegir la siguiente toma, se calcula cuanto es el coste extra que implicaria agregar dicha toma a la sesion que se este construyendo. Y de de manera voraz se selecciona la toma con mejor beneficio para el objetivo del ejercicio.

Después de construir una solución inicial, se aplica una fase de mejora local. Esta etapa establece dos posibles caminos. Por un lado, un traslado de toma de una sesion a otra. Y por otro lado, intercambiar tomas entre dos sesiones. Esto se realiza posterior al encuentro de un vecino optimo en el espacio de soluciones. Por lo que la busqueda y la accion optima se da de forma iterativa.

Por tanto, el diseño final combina tres ideas fundamentales: una construcción adaptativa basada en los costos del proyecto, un componente aleatoria para explorar distintas regiones del espacio de soluciones y una busqueda local para obtener una solución optima. Esta combinación hace que la técnica sea retadora de implementar pero a su vez apropiedad para trabajo practico en cuestion.

In [33]:
import random
import math
import numpy as np
import pandas as pd

# ============================================================
# MATRIZ TOMA - ACTOR (datos fijos)
# ============================================================

data = [
    [1,1,1,1,1,0,0,0,0,0],
    [0,0,1,1,1,0,0,0,0,0],
    [0,1,0,0,1,0,1,0,0,0],
    [1,1,0,0,0,0,1,1,0,0],
    [0,1,0,1,0,0,0,1,0,0],
    [1,1,0,1,1,0,0,0,0,0],
    [1,1,0,1,1,0,0,0,0,0],
    [1,1,0,0,0,1,0,0,0,0],
    [1,1,0,1,0,0,0,0,0,0],
    [1,1,0,0,0,1,0,0,1,0],
    [1,1,1,0,1,0,0,1,0,0],
    [1,1,1,1,0,1,0,0,0,0],
    [1,0,0,1,1,0,0,0,0,0],
    [1,0,1,0,0,1,0,0,0,0],
    [1,1,0,0,0,0,1,0,0,0],
    [0,0,0,1,0,0,0,0,0,1],
    [1,0,1,0,0,0,0,0,0,0],
    [0,0,1,0,0,1,0,0,0,0],
    [1,0,1,0,0,0,0,0,0,0],
    [1,0,1,1,1,0,0,0,0,0],
    [0,0,0,0,0,1,0,1,0,0],
    [1,1,1,1,0,0,0,0,0,0],
    [1,0,1,0,0,0,0,0,0,0],
    [0,0,1,0,0,1,0,0,0,0],
    [1,1,0,1,0,0,0,0,0,1],
    [1,0,1,0,1,0,0,0,1,0],
    [0,0,0,1,1,0,0,0,0,0],
    [1,0,0,1,0,0,0,0,0,0],
    [1,0,0,0,1,1,0,0,0,0],
    [1,0,0,1,0,0,0,0,0,0],
]

# Teniendo en cuenta la matriz brindada por el archivo se establece un df para tener una estructura inicial de datos
df = pd.DataFrame(
    data,
    index=[f"Toma_{i+1}" for i in range(30)],
    columns=[f"Actor_{i+1}" for i in range(10)]
)
df.index.name = "Toma"

display(df.head())

,Actor_1,Actor_2,Actor_3,Actor_4,Actor_5,Actor_6,Actor_7,Actor_8,Actor_9,Actor_10
Toma,,,,,,,,,,
Toma_1,1,1,1,1,1,0,0,0,0,0
Toma_2,0,0,1,1,1,0,0,0,0,0
Toma_3,0,1,0,0,1,0,1,0,0,0
Toma_4,1,1,0,0,0,0,1,1,0,0
Toma_5,0,1,0,1,0,0,0,1,0,0


In [9]:
# Se estructuran los datos en un diccionario. Para tener accesos mediante aspectos de llave-valor
def dataframe_a_diccionario(df: pd.DataFrame) -> dict:
    return {idx: row.to_numpy() for idx, row in df.iterrows()}

tomas_dict = dataframe_a_diccionario(df)
display(tomas_dict)

{'Toma_1': array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]),
 'Toma_2': array([0, 0, 1, 1, 1, 0, 0, 0, 0, 0]),
 'Toma_3': array([0, 1, 0, 0, 1, 0, 1, 0, 0, 0]),
 'Toma_4': array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0]),
 'Toma_5': array([0, 1, 0, 1, 0, 0, 0, 1, 0, 0]),
 'Toma_6': array([1, 1, 0, 1, 1, 0, 0, 0, 0, 0]),
 'Toma_7': array([1, 1, 0, 1, 1, 0, 0, 0, 0, 0]),
 'Toma_8': array([1, 1, 0, 0, 0, 1, 0, 0, 0, 0]),
 'Toma_9': array([1, 1, 0, 1, 0, 0, 0, 0, 0, 0]),
 'Toma_10': array([1, 1, 0, 0, 0, 1, 0, 0, 1, 0]),
 'Toma_11': array([1, 1, 1, 0, 1, 0, 0, 1, 0, 0]),
 'Toma_12': array([1, 1, 1, 1, 0, 1, 0, 0, 0, 0]),
 'Toma_13': array([1, 0, 0, 1, 1, 0, 0, 0, 0, 0]),
 'Toma_14': array([1, 0, 1, 0, 0, 1, 0, 0, 0, 0]),
 'Toma_15': array([1, 1, 0, 0, 0, 0, 1, 0, 0, 0]),
 'Toma_16': array([0, 0, 0, 1, 0, 0, 0, 0, 0, 1]),
 'Toma_17': array([1, 0, 1, 0, 0, 0, 0, 0, 0, 0]),
 'Toma_18': array([0, 0, 1, 0, 0, 1, 0, 0, 0, 0]),
 'Toma_19': array([1, 0, 1, 0, 0, 0, 0, 0, 0, 0]),
 'Toma_20': array([1, 0, 1, 1, 1, 0, 0, 

In [24]:
# Esta función calcula el conjunto de actores que participan en una sesión (día).
# Recibe una lista de tomas y devuelve un vector binario donde cada posición
# indica si un actor participa en al menos una de las tomas de la sesión.
# En caso de que la sesión esté vacía, se retorna un vector de ceros.

def actores_union_sesion(sesion: list, tomas_dict: dict) -> np.ndarray:
    if not sesion:
        n_actores = len(next(iter(tomas_dict.values())))
        return np.zeros(n_actores, dtype=int)

    acumulado = np.zeros(len(next(iter(tomas_dict.values()))), dtype=int)
    for toma in sesion:
        acumulado = np.logical_or(acumulado, tomas_dict[toma]).astype(int)
    return acumulado



In [25]:
# El coste total es la suma de los costes diarios
def costo_total(solucion: list, tomas_dict: dict, salario_por_actor: int = 1) -> int:
    return sum(costo_sesion(sesion, tomas_dict, salario_por_actor) for sesion in solucion if sesion)

def costo_sesion(sesion: list, tomas_dict: dict, salario_por_actor: int = 1) -> int:
    return int(actores_union_sesion(sesion, tomas_dict).sum() * salario_por_actor)

In [13]:
# Implementación de restriccioón.
# Se comprueban tres cosas:

# 1. Ninguna sesión supera el máximo de 6 tomas.
# 2. Todas las tomas aparecen en la solución.
# 3. Ninguna toma se repite.

def solucion_valida(solucion: list, tomas_dict: dict, max_tomas_por_dia: int = 6) -> bool:
    todas_las_tomas = set(tomas_dict.keys())
    vistas = []

    for sesion in solucion:
        if len(sesion) > max_tomas_por_dia:
            return False
        vistas.extend(sesion)

    return set(vistas) == todas_las_tomas and len(vistas) == len(todas_las_tomas)

In [26]:
# Durante la etapa de desarrollo. Se descubrio existia una posibilidad de que al manejar las tomas y moverlas entre sesioens surgiera una sesion vacia.
# Por lo que se limpian para evirtar ruido en el analisis GRASP
def limpiar_solucion(solucion: list) -> list:
    return [sesion[:] for sesion in solucion if sesion]

In [34]:
# Función para estructurar mejor la solucion final de forma tabular.

def resumen_solucion(solucion: list, tomas_dict: dict, salario_por_actor: int = 1) -> pd.DataFrame:
    filas = []
    total = 0

    for i, sesion in enumerate(solucion, start=1):
        c = costo_sesion(sesion, tomas_dict, salario_por_actor)
        total += c
        filas.append({
            "Sesion": i,
            "Num_Tomas": len(sesion),
            "Costo": c,
            "Tomas": ", ".join(sesion)
        })

    filas.append({
        "Sesion": "TOTAL",
        "Num_Tomas": sum(len(s) for s in solucion),
        "Costo": total,
        "Tomas": ""
    })

    return pd.DataFrame(filas)

In [16]:
# Calculo del coste extra que representa el dia si se establece una toma nueva a ese dia en cuestión

def incremento_costo_al_agregar_toma(
    sesion: list,
    toma: str,
    tomas_dict: dict,
    salario_por_actor: int = 1
) -> int:
    costo_actual = costo_sesion(sesion, tomas_dict, salario_por_actor)
    costo_nuevo = costo_sesion(sesion + [toma], tomas_dict, salario_por_actor)
    return costo_nuevo - costo_actual

In [28]:
# Construccion de soluciones que siempre parten de un aspecto inicial distinto. Por lo que las iteraciones y validaciones tendran un punto de partida inicial diferente en cada ejecución

def construir_solucion_grasp(
    tomas_dict: dict,
    max_tomas_por_dia: int = 6,
    alpha: float = 0.3,
    salario_por_actor: int = 1,
    semilla: int | None = None
) -> list:
    if semilla is not None:
        random.seed(semilla)
        np.random.seed(semilla)

    pendientes = list(tomas_dict.keys())
    random.shuffle(pendientes)

    solucion = []

    while pendientes:
        toma_inicial = pendientes.pop()
        sesion_actual = [toma_inicial]

        while len(sesion_actual) < max_tomas_por_dia and pendientes:
            evaluaciones = []
            for toma in pendientes:
                inc = incremento_costo_al_agregar_toma(
                    sesion_actual, toma, tomas_dict, salario_por_actor
                )
                evaluaciones.append((toma, inc))

            costos = [inc for _, inc in evaluaciones]
            c_min = min(costos)
            c_max = max(costos)

            umbral = c_min + alpha * (c_max - c_min)

            rcl = [toma for toma, inc in evaluaciones if inc <= umbral]
            siguiente = random.choice(rcl)

            sesion_actual.append(siguiente)
            pendientes.remove(siguiente)

        solucion.append(sesion_actual)

    return limpiar_solucion(solucion)

In [18]:
# Realiza movimientos acorde al dato mejor encontrado en la comparativa. Esto solo implica un movimiento de una toma hacia una sesion nueva
def generar_vecinos_relocate(solucion: list, max_tomas_por_dia: int = 6):
    n = len(solucion)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if not solucion[i]:
                continue
            if len(solucion[j]) >= max_tomas_por_dia:
                continue

            for idx_toma in range(len(solucion[i])):
                vecino = [s[:] for s in solucion]
                toma = vecino[i].pop(idx_toma)
                vecino[j].append(toma)
                yield limpiar_solucion(vecino)

In [29]:
# Realiza intercambios entre sesiones para construir un nuevo espacio de soluciones posibles.
def generar_vecinos_swap(solucion: list):
    n = len(solucion)

    for i in range(n):
        for j in range(i + 1, n):
            if not solucion[i] or not solucion[j]:
                continue

            for idx_a in range(len(solucion[i])):
                for idx_b in range(len(solucion[j])):
                    vecino = [s[:] for s in solucion]
                    vecino[i][idx_a], vecino[j][idx_b] = vecino[j][idx_b], vecino[i][idx_a]
                    yield limpiar_solucion(vecino)

In [35]:
# Ejecuta de forma constante una busqueda de la solución optima acorde a un punto de salida inicialmente aleatorio
# Genera vecinos o un espacio de soluciones para asi realizar los cambios pertinentes en una busqueda de soluciones locales.
def busqueda_local(
    solucion_inicial: list,
    tomas_dict: dict,
    max_tomas_por_dia: int = 6,
    salario_por_actor: int = 1,
    max_iter_sin_mejora: int = 100
) -> list:
    actual = limpiar_solucion(solucion_inicial)
    mejor_costo = costo_total(actual, tomas_dict, salario_por_actor)
    sin_mejora = 0

    while sin_mejora < max_iter_sin_mejora:
        mejorado = False

        for vecino in generar_vecinos_relocate(actual, max_tomas_por_dia):
            if not solucion_valida(vecino, tomas_dict, max_tomas_por_dia):
                continue

            c = costo_total(vecino, tomas_dict, salario_por_actor)
            if c < mejor_costo:
                actual = vecino
                mejor_costo = c
                mejorado = True
                sin_mejora = 0
                break

        if mejorado:
            continue

        for vecino in generar_vecinos_swap(actual):
            if not solucion_valida(vecino, tomas_dict, max_tomas_por_dia):
                continue

            c = costo_total(vecino, tomas_dict, salario_por_actor)
            if c < mejor_costo:
                actual = vecino
                mejor_costo = c
                mejorado = True
                sin_mejora = 0
                break

        if not mejorado:
            sin_mejora += 1

    return limpiar_solucion(actual)

In [36]:

def grasp(
    tomas_dict: dict,
    max_tomas_por_dia: int = 6,
    alpha: float = 0.3,
    iteraciones: int = 100,
    salario_por_actor: int = 1,
    max_iter_sin_mejora_local: int = 100,
    semilla: int | None = None
):
    if semilla is not None:
        random.seed(semilla)
        np.random.seed(semilla)

    mejor_solucion = None
    mejor_costo = math.inf
    historial = []

    for it in range(iteraciones):
        solucion_inicial = construir_solucion_grasp(
            tomas_dict=tomas_dict,
            max_tomas_por_dia=max_tomas_por_dia,
            alpha=alpha,
            salario_por_actor=salario_por_actor
        )

        solucion_mejorada = busqueda_local(
            solucion_inicial=solucion_inicial,
            tomas_dict=tomas_dict,
            max_tomas_por_dia=max_tomas_por_dia,
            salario_por_actor=salario_por_actor,
            max_iter_sin_mejora=max_iter_sin_mejora_local
        )

        costo = costo_total(solucion_mejorada, tomas_dict, salario_por_actor)
        historial.append(costo)

        if costo < mejor_costo:
            mejor_costo = costo
            mejor_solucion = limpiar_solucion(solucion_mejorada)

    return {
        "mejor_solucion": mejor_solucion,
        "mejor_costo": mejor_costo,
        "historial_costos": historial
    }

In [38]:
resultado = grasp(
    tomas_dict=tomas_dict,
    max_tomas_por_dia=6,
    alpha=0.25,
    iteraciones=200,
    salario_por_actor=1,
    max_iter_sin_mejora_local=80,
    semilla=42
)

print("Mejor coste encontrado:", resultado["mejor_costo"])
print("\nMejor solución:")

for i, sesion in enumerate(resultado["mejor_solucion"], start=1):
    print(f"Sesión {i}: {sesion} | Coste = {costo_sesion(sesion, tomas_dict)}")

print("\nResumen:")
display(resumen_solucion(resultado["mejor_solucion"], tomas_dict))

Mejor coste encontrado: 27

Mejor solución:
Sesión 1: ['Toma_18', 'Toma_19', 'Toma_24', 'Toma_14', 'Toma_23', 'Toma_17'] | Coste = 3
Sesión 2: ['Toma_29', 'Toma_8', 'Toma_21', 'Toma_4', 'Toma_15', 'Toma_3'] | Coste = 6
Sesión 3: ['Toma_27', 'Toma_28', 'Toma_13', 'Toma_30', 'Toma_20', 'Toma_2'] | Coste = 4
Sesión 4: ['Toma_7', 'Toma_22', 'Toma_9', 'Toma_25', 'Toma_16', 'Toma_6'] | Coste = 6
Sesión 5: ['Toma_12', 'Toma_10', 'Toma_5', 'Toma_1', 'Toma_11', 'Toma_26'] | Coste = 8

Resumen:


,Sesion,Num_Tomas,Costo,Tomas
0,1,6,3,"Toma_18, Toma_19, Toma_24, Toma_14, Toma_23, T..."
1,2,6,6,"Toma_29, Toma_8, Toma_21, Toma_4, Toma_15, Toma_3"
2,3,6,4,"Toma_27, Toma_28, Toma_13, Toma_30, Toma_20, T..."
3,4,6,6,"Toma_7, Toma_22, Toma_9, Toma_25, Toma_16, Toma_6"
4,5,6,8,"Toma_12, Toma_10, Toma_5, Toma_1, Toma_11, Tom..."
5,TOTAL,30,27,
